In [57]:
from utils import get_dataset_lines

# Construct the Graph of a Spectrum

**Code Challenge**: Construct the graph of a spectrum.

**Input**: A space-delimited list of integers $Spectrum$. 

**Output**: $Graph(Spectrum)$.

**Note**: Throughout this chapter, all dataset problems implicitly use the [standard integer-valued mass table](integer_mass_table.txt) for the regular twenty amino acids. Examples sometimes use the toy amino acid alphabet $\{X, Z\}$ whose masses are 4 and 5, respectively.

**Sample Input**:

```
57 71 154 185 301 332 415 429 486
```

**Sample Output**:

```
0->57:G
0->71:A
57->154:P
57->185:K
71->185:N
154->301:F
185->332:F
301->415:N
301->429:K
332->429:P
415->486:A
429->486:G
```

In [58]:
def get_mass_table(toy=False):
    # Read the mass table from the file 'integer_mass_table.txt'
    mass_table = {}
    with open('integer_mass_table.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split()
            aa = parts[0]
            mass = int(parts[1])
            # Key = Mass, Value = Amino Acid
            if mass not in mass_table:
                mass_table[mass] = aa
    
    if toy:
        mass_table[4] = 'X'
        mass_table[5] = 'Z'
                
    return mass_table

def get_aa_to_mass_table(toy=False):
    aa_map = {}
    with open('integer_mass_table.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split()
            aa = parts[0]
            mass = int(parts[1])
            aa_map[aa] = mass
            
    if toy:
        aa_map['X'] = 4
        aa_map['Z'] = 5
    return aa_map

In [59]:

def ConstructSpectrumGraph(spectrum, toy=False):
    mass_table = get_mass_table(toy=toy)
    nodes = [0] + sorted(spectrum)
    edges = []
    
    for i in range(len(nodes)):
        u = nodes[i]
        for j in range(i + 1, len(nodes)):
            v = nodes[j]
            diff = v - u
            
            if diff in mass_table:
                edges.append(f"{u}->{v}:{mass_table[diff]}")
                
    return edges

In [60]:
# Sample Input
spectrum = [57, 71, 154, 185, 301, 332, 415, 429, 486]

# Run the function
result_edges = ConstructSpectrumGraph(spectrum)

# Print the result
for edge in result_edges:
    print(edge)

# Test Assertion
expected_output = [
    "0->57:G", "0->71:A", "57->154:P", "57->185:K", "71->185:N",
    "154->301:F", "185->332:F", "301->415:N", "301->429:K",
    "332->429:P", "415->486:A", "429->486:G"
]
assert result_edges == expected_output, f"Expected {expected_output}, but got {result_edges}"
print("Test passed!")

0->57:G
0->71:A
57->154:P
57->185:K
71->185:N
154->301:F
185->332:F
301->415:N
301->429:K
332->429:P
415->486:A
429->486:G
Test passed!


In [61]:
# Test Dataset
test_dataset_filename = 'dataset_30262_5.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    spectrum = list(map(int, lines[0].split()))
    
    result_edges = ConstructSpectrumGraph(spectrum)
    for edge in result_edges:
        print(edge)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

0->99:V
0->99:V
99->200:T
99->213:N
99->200:T
99->213:N
200->297:P
213->369:R
297->483:W
369->483:N
369->506:H
483->614:M
506->577:A
577->691:N
614->715:T
691->748:G
691->847:R
715->871:R
748->847:V
847->918:A
871->999:K
918->1046:K
999->1130:M
1046->1143:P
1130->1261:M
1143->1256:I
1256->1393:H
1261->1375:N
1375->1512:H
1393->1507:N
1507->1638:M
1512->1625:I
1625->1722:P
1638->1769:M
1722->1850:K
1769->1897:K
1850->1921:A
1897->2053:R
1921->2020:V
1921->2077:R
2020->2077:G
2053->2154:T
2077->2191:N
2154->2285:M
2191->2262:A
2262->2399:H
2285->2399:N
2285->2471:W
2399->2555:R
2471->2568:P
2555->2669:N
2555->2669:N
2568->2669:T
2568->2669:T
2669->2768:V
2669->2768:V


# Decoding an Ideal Spectrum

**Code Challenge**: Solve the Decoding an Ideal Spectrum Problem.

**Input**: A space-delimited list of integers $Spectrum$.

**Output**: An amino acid string that explains $Spectrum$.

**Sample Input**:

```
57 71 154 185 301 332 415 429 486
```

**Sample Output**:

```
GPFNA
```

In [62]:
def DecodeIdealSpectrum(spectrum):
    spectrum = sorted(list(set(spectrum)))
    max_mass = spectrum[-1]
    
    # Reuse ConstructSpectrumGraph to build the graph
    graph_edges = ConstructSpectrumGraph(spectrum)
    
    # Parse graph edges into Adjacency List
    adj = {}
    for edge in graph_edges:
        # Format: "u->v:aa"
        u_str, rest = edge.split("->")
        v_str, aa = rest.split(":")
        u = int(u_str)
        v = int(v_str)
        
        if u not in adj:
            adj[u] = []
        adj[u].append((v, aa))
                
    # Helper to check spectrum match
    def check_spectrum(path_masses):
        # path_masses includes 0 and max_mass
        # Ideal Spectrum = {Prefix Masses} U {Suffix Masses}
        # Suffix Mass = Total Mass - Prefix Mass
        ideal = set(path_masses)
        ideal.update(max_mass - m for m in path_masses)
        
        # Remove 0 if it is not in the original spectrum (usually it isn't based on sample)
        if len(spectrum) > 0 and 0 not in spectrum:
            ideal.discard(0)
            
        return sorted(list(ideal)) == spectrum

    # DFS to find path
    # path_masses: list of visited masses (prefixes)
    def dfs(current_mass, path_string, path_masses):
        if current_mass == max_mass:
            if check_spectrum(path_masses):
                return path_string
            return None
        
        if current_mass in adj:
            for next_mass, aa in adj[current_mass]:
                result = dfs(next_mass, path_string + aa, path_masses + [next_mass])
                if result:
                    return result
        return None

    path = dfs(0, "", [0])
    
    # Change all K to Q and I to L as required by solution checker
    if path:
        return path.replace('K', 'Q').replace('I', 'L')
    return None

In [63]:
# Sample Input
spectrum = [57, 71, 154, 185, 301, 332, 415, 429, 486]

# Run
result = DecodeIdealSpectrum(spectrum)
print(result)

# Assert
expected_output = "GPFNA"
assert result == expected_output, f"Expected {expected_output}, but got {result}"
print("Test passed!")

GPFNA
Test passed!


In [64]:
# Test Dataset
test_dataset_filename = 'dataset_30262_8.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    spectrum = list(map(int, lines[0].split()))
    
    result = DecodeIdealSpectrum(spectrum)
    print(result)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

NRPGFYTQFMLDPGATVFMLQY


# Converting a Peptide into a Peptide Vector Problem

Convert a peptide into a peptide vector.

**Code Challenge**: Solve the Converting a Peptide into a Peptide Vector Problem.

**Input**: An amino acid string $P$.

**Output**: The peptide vector of $P$ (in the form of space-separated integers).

**Sample Input**:

```
XZZXX
```

**Sample Output**:

```
0 0 0 1 0 0 0 0 1 0 0 0 0 1 0 0 0 1 0 0 0 1
```

In [65]:
def PeptideToVector(P, toy=False):
    aa_map = get_aa_to_mass_table(toy=toy)
    
    prefix_masses = []
    current_mass = 0
    for aa in P:
        current_mass += aa_map[aa]
        prefix_masses.append(current_mass)
        
    total_mass = prefix_masses[-1]
    vector = [0] * total_mass
    
    for m in prefix_masses:
        vector[m-1] = 1 # 1-based index to 0-based
        
    return vector

In [66]:
# Sample Input
P = "XZZXX"

vector = PeptideToVector(P, toy=True)
result = " ".join(map(str, vector))
print(result)

# Test Assertion
expected_output = "0 0 0 1 0 0 0 0 1 0 0 0 0 1 0 0 0 1 0 0 0 1"
assert result == expected_output
print("Test passed!")

0 0 0 1 0 0 0 0 1 0 0 0 0 1 0 0 0 1 0 0 0 1
Test passed!


In [67]:
# Test Dataset
test_dataset_filename = 'dataset_30264_5.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    P = lines[0].strip()
    
    vector = PeptideToVector(P)
    print(" ".join(map(str, vector)))
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found.")
except Exception as e:
    print(f"An error occurred: {e}")


0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 

# Converting a Peptide Vector into a Peptide Problem

Since a peptide vector uniquely defines the peptide that it originated from, we will use the terms "peptide vector" and "peptide" interchangeably.

Convert a peptide vector into a peptide.

**Code Challenge**: Solve the Converting a Peptide Vector into a Peptide Problem.

**Input**: A space-delimited binary vector $P$.

**Output**: An amino acid string whose binary peptide vector matches $P$. For masses with more than one amino acid, any choice may be used.

**Sample Input**:

```
0 0 0 1 0 0 0 0 1 0 0 0 0 1 0 0 0 1 0 0 0 1
```

**Sample Output**:

```
XZZXX
```

In [68]:
def VectorToPeptide(vector, toy=False):
    # Map mass -> aa
    mass_to_aa = get_mass_table(toy=toy)
    
    peptide = ""
    last_mass = 0
    
    for i in range(len(vector)):
        if vector[i] == 1:
            current_mass = i + 1
            diff = current_mass - last_mass
            
            if diff in mass_to_aa:
                peptide += mass_to_aa[diff]
                last_mass = current_mass
            else:
                # If we encounter a mass difference that isn't in our table,
                # it might be a problem or we just skip (though problem implies valid inputs)
                pass
                
    return peptide

In [69]:
# Sample Input
vector_input = "0 0 0 1 0 0 0 0 1 0 0 0 0 1 0 0 0 1 0 0 0 1"
vector = list(map(int, vector_input.split()))

# Run
result = VectorToPeptide(vector, toy=True)
print(result)

# Test Assertion
expected_output = "XZZXX"
assert result == expected_output
print("Test passed!")

XZZXX
Test passed!


In [70]:
# Test Dataset
test_dataset_filename = 'dataset_30264_6.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    vector = list(map(int, lines[0].split()))
    
    result = VectorToPeptide(vector)
    print(result)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found.")
except Exception as e:
    print(f"An error occurred: {e}")

SSEARKKNTRSSIHRKWRHV


# Peptide Sequencing Problem

Any path connecting source to sink in this DAG corresponds to an amino acid string $Peptide$, and the total weight of nodes on this path is equal to $Score(Peptide', Spectrum')$. We have therefore reduced the Peptide Sequencing Problem to the problem of finding a maximum-weight path from source to sink in a node-weighted DAG.

**Code Challenge**: Solve the Peptide Sequencing Problem.

**Input**: A space-delimited spectral vector $Spectrum'$.

**Output**: An amino acid string with maximum score against $Spectrum'$. For masses with more than one amino acid, any choice may be used.

**Note**: When a spectral vector $Spectrum' = s_1 \dots s_m$ is given, it does not have a zero-th element; in your implementations, you should assume that $s_0$ is equal to zero.

**Sample Input**:

```
0 0 0 4 -2 -3 -1 -7 6 5 3 2 1 9 3 -8 0 3 1 2 1 8
```

**Sample Output**:

```
XZZXX
```

In [71]:
def PeptideSequencing(spectral_vector, toy=False):
    # Map mass -> aa
    mass_to_aa = get_mass_table(toy=toy)
    
    n = len(spectral_vector)
    
    # Initialize DP array
    # dp[i] stores the max score to reach mass i
    import sys
    min_score = -sys.maxsize
    dp = [min_score] * (n + 1)
    dp[0] = 0
    
    # Backtrack pointer: mass -> (prev_mass, aa)
    backtrack = {}
    
    for i in range(1, n + 1):
        # We want to find best edge (prev -> i)
        # edge exists if i - prev = mass of some aa
        # Weight of node i is spectral_vector[i-1]
        
        current_node_weight = spectral_vector[i-1]
        
        best_prev_score = min_score
        best_aa = None
        best_prev_mass = -1
        
        for mass, aa in mass_to_aa.items():
            prev = i - mass
            if prev >= 0:
                if dp[prev] != min_score:
                    if dp[prev] > best_prev_score:
                        best_prev_score = dp[prev]
                        best_aa = aa
                        best_prev_mass = prev
        
        if best_prev_score != min_score:
            dp[i] = best_prev_score + current_node_weight
            backtrack[i] = (best_prev_mass, best_aa)
            
    # Reconstruct
    peptide = ""
    curr = n
    if dp[n] == min_score:
        return "" # Should not happen for valid inputs
        
    while curr > 0:
        prev_mass, aa = backtrack[curr]
        peptide = aa + peptide
        curr = prev_mass
        
    return peptide

In [72]:
# Sample Input
input_vector_str = "0 0 0 4 -2 -3 -1 -7 6 5 3 2 1 9 3 -8 0 3 1 2 1 8"
spectral_vector = list(map(int, input_vector_str.split()))

# Run
result = PeptideSequencing(spectral_vector, toy=True)
print(result)

# Assert
expected_output = "XZZXX"
assert result == expected_output, f"Expected {expected_output}, but got {result}"
print("Test passed!")

XZZXX
Test passed!


In [73]:
# Test Dataset
test_dataset_filename = 'dataset_30264_13.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    spectral_vector = list(map(int, lines[0].split()))
    
    result = PeptideSequencing(spectral_vector)
    print(result)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found.")
except Exception as e:
    print(f"An error occurred: {e}")

AGGHVGGAGV
